# RAG Generation and Evaluation Pipeline

This Colab notebook runs the complete workflow:

1. Read questions from `generations.jsonl`.
2. Generate answers with Vertex AI RAG Engine.
3. Evaluate the generated answers against `gold_answers.json` with Gemini 2.5 Flash.
4. Evaluate and a final score.

Required input files:

- `generations.jsonl`
- `gold_answers.json`


## 1. Install Required Libraries

This cell installs the libraries required for Vertex AI, Gemini, RAG Engine, and progress bars.

In [ ]:
!pip install -q google-cloud-aiplatform google-genai google-auth tqdm

## 2. Authenticate with Google Cloud

This cell connects Colab directly to your Google account. No service account key is required.

Make sure that your Google account has access to the Google Cloud project and to the RAG corpus.

In [20]:
from google.colab import auth
import google.auth

auth.authenticate_user()

credentials, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

print("Google Cloud authentication completed.")

Google Cloud authentication completed.


## 3. Upload Input Files

Upload:

- `generations.jsonl`, containing the questions;
- `gold_answers.json`, containing the reference answers.

In [22]:
from google.colab import files

print("Upload generations.jsonl:")
uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded.keys():
    print("-", filename)

Upload generations.jsonl:


Saving generations.jsonl to generations.jsonl
Uploaded files:
- generations.jsonl


In [37]:
print("Uploadgold_answers.json:")
uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded.keys():
    print("-", filename)

Uploadgold_answers.json:


Saving gold_answers.json to gold_answers.json
Uploaded files:
- gold_answers.json


## 4. Import Python Libraries

These imports are used for file processing, RAG retrieval, Gemini generation, and evaluation.

In [28]:
import json
import time
from pathlib import Path

from tqdm import tqdm

from google import genai
from google.genai import types

import vertexai
from vertexai import rag

## 5. Configuration

Edit these values if your project, corpus, location, or models are different.

In [39]:
# Google Cloud project used for the lab.
PROJECT_ID = "YOUR_PROJECT_ID"

# Region where the RAG corpus was created.
RAG_LOCATION = "europe-west6"

# Location used by Gemini through Vertex AI.
GENAI_LOCATION = "global"

# Model used to generate answers with RAG grounding.
GENERATION_MODEL_ID = "gemini-2.5-flash-lite"

# Model used as an LLM-as-a-Judge.
JUDGE_MODEL_ID = "gemini-2.5-flash-lite"

# Full resource name of the RAG corpus created in Vertex AI RAG Engine. (Google Cloud platform -> RAG Engine -> rag-fal-corpus -> Details -> Resource name)
RAG_CORPUS_NAME = "YOUR_CORPUS_URI"

# Input files.
QUESTIONS_JSONL = "/content/generations.jsonl"
GOLD_ANSWERS_JSON = "/content/gold_answers.json"

# Output files.
GENERATIONS_FILLED_JSONL = "/content/generations_filled.jsonl"
RAG_EVALUATIONS_JSONL = "/content/rag_evaluations.jsonl"
PIPELINE_SUMMARY_JSON = "/content/rag_pipeline_summary.json"

# Retrieval parameters.
TOP_K = 10
VECTOR_DISTANCE_THRESHOLD = 0.7

## 6. Initialize Vertex AI and Gemini Clients

This cell initializes Vertex AI and creates a Gemini client.

In [40]:
vertexai.init(
    project=PROJECT_ID,
    location=RAG_LOCATION,
    credentials=credentials,
)

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=GENAI_LOCATION,
    credentials=credentials,
)

print("Vertex AI and Gemini clients initialized.")

Vertex AI and Gemini clients initialized.


## 7. Utility Functions

These functions load files, count JSONL rows, parse JSON outputs, and format retrieved chunks.

In [41]:
def count_non_empty_lines(path: Path) -> int:
    """Count non-empty lines in a JSONL file."""
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def load_gold_answers(path: Path) -> dict:
    """Load reference answers from gold_answers.json."""
    if not path.exists():
        raise FileNotFoundError(f"Gold answers file not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def extract_json_from_model_output(text: str) -> dict:
    """Parse JSON returned by Gemini, including markdown-wrapped JSON."""
    text = text.strip()

    if text.startswith("```json"):
        text = text.removeprefix("```json").strip()
    if text.startswith("```"):
        text = text.removeprefix("```").strip()
    if text.endswith("```"):
        text = text.removesuffix("```").strip()

    return json.loads(text)


def safe_get_contexts(retrieval_response):
    """Safely extract contexts from a RAG retrieval response."""
    if (
        hasattr(retrieval_response, "contexts")
        and retrieval_response.contexts
        and hasattr(retrieval_response.contexts, "contexts")
        and retrieval_response.contexts.contexts
    ):
        return retrieval_response.contexts.contexts

    return []


def build_retrieved_chunks(retrieval_response, top_k: int = 10) -> list:
    """Convert retrieved RAG contexts into a clean list of chunks."""
    contexts = safe_get_contexts(retrieval_response)
    chunks = []

    for rank, ctx in enumerate(contexts[:top_k]):
        text = getattr(ctx, "text", "") or ""
        score = getattr(ctx, "score", None)
        source_uri = getattr(ctx, "source_uri", None)
        source_display_name = getattr(ctx, "source_display_name", None)

        chunks.append(
            {
                "id": f"chunk_{rank:04d}",
                "text": text,
                "metadata": {
                    "source_uri": source_uri,
                    "source_display_name": source_display_name,
                    "score": score,
                },
                "distance": score,
                "rank": rank,
            }
        )

    return chunks

## 8. RAG Retrieval

This function retrieves the most relevant chunks from the RAG corpus for a given question.

In [42]:
def retrieve_chunks(question: str):
    """Retrieve relevant chunks from the Vertex AI RAG corpus."""
    retrieval_response = rag.retrieval_query(
        rag_resources=[
            rag.RagResource(
                rag_corpus=RAG_CORPUS_NAME,
            )
        ],
        rag_retrieval_config=rag.RagRetrievalConfig(
            top_k=TOP_K,
            filter=rag.Filter(
                vector_distance_threshold=VECTOR_DISTANCE_THRESHOLD,
            ),
        ),
        text=question,
    )

    retrieved_chunks = build_retrieved_chunks(
        retrieval_response,
        top_k=TOP_K,
    )

    return retrieval_response, retrieved_chunks

## 9. RAG Answer Generation

This section defines how Gemini generates answers while using the RAG corpus as a retrieval tool.

In [43]:
def generate_rag_answer(question: str) -> str:
    """Generate an answer using Gemini connected to the RAG corpus."""
    tools = [
        types.Tool(
            retrieval=types.Retrieval(
                vertex_rag_store=types.VertexRagStore(
                    rag_resources=[
                        types.VertexRagStoreRagResource(
                            rag_corpus=RAG_CORPUS_NAME
                        )
                    ],
                    similarity_top_k=TOP_K,
                    vector_distance_threshold=VECTOR_DISTANCE_THRESHOLD,
                )
            )
        )
    ]

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=question)],
        )
    ]

    config = types.GenerateContentConfig(
        temperature=0.2,
        top_p=0.95,
        max_output_tokens=8192,
        tools=tools,
    )

    generated_parts = []

    for chunk in client.models.generate_content_stream(
        model=GENERATION_MODEL_ID,
        contents=contents,
        config=config,
    ):
        text = getattr(chunk, "text", None)

        if text:
            generated_parts.append(text)

    return "".join(generated_parts).strip()


def generate_filled_record(record: dict) -> dict:
    """Enrich one question record with RAG answer, chunks, metadata, and latency."""
    question = record.get("question", "").strip()

    if not question:
        raise ValueError("Field 'question' is missing or empty.")

    question_start = time.perf_counter()

    retrieval_start = time.perf_counter()
    retrieval_response, retrieved_chunks = retrieve_chunks(question)
    retrieval_end = time.perf_counter()

    generation_start = time.perf_counter()
    generated_answer = generate_rag_answer(question)
    generation_end = time.perf_counter()

    question_end = time.perf_counter()

    record["generated_answer"] = generated_answer
    record["retrieved_chunks"] = retrieved_chunks

    if "method_metadata" not in record or not isinstance(record["method_metadata"], dict):
        record["method_metadata"] = {}

    record["method_metadata"].update(
        {
            "provider": "vertex_ai_rag",
            "generation_model": GENERATION_MODEL_ID,
            "rag_corpus": RAG_CORPUS_NAME,
            "top_k": TOP_K,
            "vector_distance_threshold": VECTOR_DISTANCE_THRESHOLD,
        }
    )

    record["latency_ms"] = round((generation_end - retrieval_start) * 1000, 3)
    record["latency_breakdown_ms"] = {
        "retrieval_ms": round((retrieval_end - retrieval_start) * 1000, 3),
        "generation_ms": round((generation_end - generation_start) * 1000, 3),
    }
    record["question_processing_time_ms"] = round((question_end - question_start) * 1000, 3)

    return record

## 10. LLM-as-a-Judge Evaluation

Gemini 2.5 Flash compares each generated answer with the gold answer and assigns a binary score:

- `1`: the generated answer correctly answers the question;
- `0`: the generated answer is incorrect, incomplete, or not aligned.

In [45]:
def evaluate_answer_with_gemini(question: str, gold_answer: str, generated_answer: str) -> dict:
    """Evaluate whether a generated answer is aligned with the gold answer."""
    prompt = f"""
You are evaluating the answer of a RAG system.

Your task is to decide whether the GENERATED ANSWER correctly answers the QUESTION,
based on the GOLD ANSWER.

The generated answer does not need to use the exact same wording as the gold answer.
It should be considered correct if it contains the essential information needed to answer the question.

Be strict:
- If the generated answer contradicts the gold answer, score 0.
- If the generated answer misses an essential element, score 0.
- If the generated answer is too vague to verify, score 0.
- If the generated answer adds unsupported but harmless details, ignore them unless they change the meaning.
- If the generated answer is empty, score 0.

Return only valid JSON in the following format:

{{
  "aligned": "yes" or "no",
  "score": 1 or 0,
  "reason": "short explanation in French"
}}

QUESTION:
{question}

GOLD ANSWER:
{gold_answer}

GENERATED ANSWER:
{generated_answer}
""".strip()

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=prompt)],
        )
    ]

    config = types.GenerateContentConfig(
        temperature=0.0,
        top_p=1.0,
        max_output_tokens=1024,
        response_mime_type="application/json",
    )

    response = client.models.generate_content(
        model=JUDGE_MODEL_ID,
        contents=contents,
        config=config,
    )

    raw_text = response.text.strip()
    result = extract_json_from_model_output(raw_text)

    aligned = result.get("aligned", "").lower().strip()

    try:
        score = int(result.get("score", 0))
    except Exception:
        score = 0

    if aligned not in {"yes", "no"}:
        aligned = "yes" if score == 1 else "no"

    if score not in {0, 1}:
        score = 1 if aligned == "yes" else 0

    return {
        "aligned": aligned,
        "score": score,
        "reason": result.get("reason", ""),
    }

## 11. Complete Pipeline

This function executes the full workflow and writes all output files.

In [46]:
def run_complete_rag_pipeline():
    """Run the complete RAG generation and evaluation pipeline."""
    questions_path = Path(QUESTIONS_JSONL)
    gold_path = Path(GOLD_ANSWERS_JSON)
    filled_output_path = Path(GENERATIONS_FILLED_JSONL)
    evaluations_output_path = Path(RAG_EVALUATIONS_JSONL)
    summary_output_path = Path(PIPELINE_SUMMARY_JSON)

    if not questions_path.exists():
        raise FileNotFoundError(f"Questions file not found: {questions_path}")

    if not gold_path.exists():
        raise FileNotFoundError(f"Gold answers file not found: {gold_path}")

    gold_answers = load_gold_answers(gold_path)
    total_questions = count_non_empty_lines(questions_path)

    processed_count = 0
    generation_success_count = 0
    evaluation_success_count = 0
    missing_gold_count = 0
    error_count = 0
    total_score = 0
    scores = []
    processing_times_ms = []

    pipeline_start = time.perf_counter()

    with questions_path.open("r", encoding="utf-8") as fin, \
         filled_output_path.open("w", encoding="utf-8") as filled_fout, \
         evaluations_output_path.open("w", encoding="utf-8") as eval_fout:

        progress_bar = tqdm(
            fin,
            total=total_questions,
            desc="RAG generation and evaluation",
            unit="question",
        )

        for line_number, line in enumerate(progress_bar, start=1):
            line = line.strip()

            if not line:
                continue

            processed_count += 1
            question_start = time.perf_counter()

            try:
                record = json.loads(line)

                question_id = record.get("question_id")
                question = record.get("question", "")

                if not question_id:
                    raise ValueError("Missing question_id.")

                filled_record = generate_filled_record(record)

                filled_fout.write(json.dumps(filled_record, ensure_ascii=False) + "\n")
                generation_success_count += 1

                generated_answer = filled_record.get("generated_answer", "")
                gold_answer = gold_answers.get(question_id)

                if gold_answer is None:
                    missing_gold_count += 1

                    evaluation_record = {
                        "question_id": question_id,
                        "question": question,
                        "gold_answer": None,
                        "generated_answer": generated_answer,
                        "aligned": "no",
                        "score": 0,
                        "reason": "Aucune réponse de référence trouvée pour cette question.",
                        "error": "missing_gold_answer",
                    }

                else:
                    evaluation = evaluate_answer_with_gemini(
                        question=question,
                        gold_answer=gold_answer,
                        generated_answer=generated_answer,
                    )

                    score = evaluation["score"]
                    total_score += score
                    scores.append(score)
                    evaluation_success_count += 1

                    evaluation_record = {
                        "question_id": question_id,
                        "question": question,
                        "gold_answer": gold_answer,
                        "generated_answer": generated_answer,
                        "aligned": evaluation["aligned"],
                        "score": score,
                        "reason": evaluation["reason"],
                    }

                eval_fout.write(json.dumps(evaluation_record, ensure_ascii=False) + "\n")

                question_end = time.perf_counter()
                elapsed_ms = round((question_end - question_start) * 1000, 3)
                processing_times_ms.append(elapsed_ms)

                progress_bar.set_postfix(
                    question_id=question_id,
                    score=evaluation_record["score"],
                    avg_score=round(sum(scores) / len(scores), 4) if scores else 0.0,
                    last_ms=elapsed_ms,
                )

            except Exception as e:
                error_count += 1
                question_end = time.perf_counter()
                elapsed_ms = round((question_end - question_start) * 1000, 3)
                processing_times_ms.append(elapsed_ms)

                error_record = {
                    "line_number": line_number,
                    "error": str(e),
                    "raw_line": line,
                    "aligned": "no",
                    "score": 0,
                    "question_processing_time_ms": elapsed_ms,
                }

                eval_fout.write(json.dumps(error_record, ensure_ascii=False) + "\n")

                progress_bar.set_postfix(
                    error=f"line {line_number}",
                    avg_score=round(sum(scores) / len(scores), 4) if scores else 0.0,
                    last_ms=elapsed_ms,
                )

    pipeline_end = time.perf_counter()
    total_time_ms = round((pipeline_end - pipeline_start) * 1000, 3)

    average_time_ms = (
        round(sum(processing_times_ms) / len(processing_times_ms), 3)
        if processing_times_ms
        else 0.0
    )

    final_score = (
        round(total_score / evaluation_success_count, 4)
        if evaluation_success_count > 0
        else 0.0
    )

    summary = {
        "project_id": PROJECT_ID,
        "rag_corpus": RAG_CORPUS_NAME,
        "generation_model": GENERATION_MODEL_ID,
        "judge_model": JUDGE_MODEL_ID,
        "questions_file": str(questions_path),
        "gold_answers_file": str(gold_path),
        "generated_answers_file": str(filled_output_path),
        "evaluations_file": str(evaluations_output_path),
        "processed_count": processed_count,
        "generation_success_count": generation_success_count,
        "evaluation_success_count": evaluation_success_count,
        "missing_gold_count": missing_gold_count,
        "error_count": error_count,
        "total_score": total_score,
        "final_score": final_score,
        "final_score_percent": round(final_score * 100, 2),
        "total_time_ms": total_time_ms,
        "average_time_ms": average_time_ms,
        "processing_times_ms": processing_times_ms,
    }

    with summary_output_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\n--- Final RAG Pipeline Summary ---")
    print(f"Questions processed          : {processed_count}")
    print(f"Generation successes         : {generation_success_count}")
    print(f"Evaluation successes         : {evaluation_success_count}")
    print(f"Missing gold answers         : {missing_gold_count}")
    print(f"Errors                       : {error_count}")
    print(f"Total score                  : {total_score}/{evaluation_success_count}")
    print(f"Final score                  : {final_score}")
    print(f"Final score percent          : {round(final_score * 100, 2)}%")
    print(f"Generated answers file       : {filled_output_path.resolve()}")
    print(f"Evaluations file             : {evaluations_output_path.resolve()}")
    print(f"Pipeline summary file        : {summary_output_path.resolve()}")

## 12. Run the Pipeline

Execute this cell to launch the complete RAG generation and evaluation process.

In [47]:
run_complete_rag_pipeline()

RAG generation and evaluation: 100%|██████████| 20/20 [01:16<00:00,  3.83s/question, avg_score=0.65, last_ms=8.43e+3, question_id=q_0019, score=0]


--- Final RAG Pipeline Summary ---
Questions processed          : 20
Generation successes         : 20
Evaluation successes         : 20
Missing gold answers         : 0
Errors                       : 0
Total score                  : 13/20
Final score                  : 0.65
Final score percent          : 65.0%
Generated answers file       : /content/generations_filled.jsonl
Evaluations file             : /content/rag_evaluations.jsonl
Pipeline summary file        : /content/rag_pipeline_summary.json
